# Lab 7 - Logistic regression regularization


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

import pandas as pd
from scipy.special import expit
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


## Task 1 - prostate data: lasso, ridge, elastic net


In [2]:
X_prostate = pd.read_csv("../../tasks/datasets/prostate_x.txt", sep=r"\s+", index_col=0)
y_prostate = pd.read_csv("../../tasks/datasets/prostate_y.txt", sep=r"\s+", index_col=0).iloc[:, 0].to_numpy()
print(X_prostate.shape, y_prostate.shape, np.bincount(y_prostate.astype(int)))


(102, 6033) (102,) [50 52]


In [ ]:
X = X_prostate.to_numpy()
y = y_prostate.astype(int)
Cs = np.logspace(-2, 2, 12)  # C = 1 / lambda
penalties = {
    "ridge": {"penalty": "l2", "solver": "saga", "l1_ratio": None},
    "lasso": {"penalty": "l1", "solver": "saga", "l1_ratio": None},
    "elastic_net": {"penalty": "elasticnet", "solver": "saga", "l1_ratio": 0.5},
}

paths = {}
for name, cfg in penalties.items():
    coefs = []
    for C in Cs:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE, **cfg),
        )
        clf.fit(X, y)
        coefs.append(clf.named_steps["logisticregression"].coef_[0])
    paths[name] = np.asarray(coefs)
    selected = np.sum(np.abs(paths[name][-1]) > 1e-6)
    print(name, "nonzero coefficients at largest C:", selected)


ridge nonzero coefficients at largest C: 6031


In [ ]:
top_features = np.argsort(np.abs(paths["lasso"][-1]))[-10:]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, coefs) in zip(axes, paths.items()):
    for j in top_features:
        ax.plot(1 / Cs, coefs[:, j], label=f"V{j+1}" if name == "lasso" else None)
    ax.set_xscale("log")
    ax.set_title(name)
    ax.set_xlabel("lambda = 1/C")
    ax.grid(True)
axes[0].set_ylabel("coefficient")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
cv_models = {}
for name, cfg in penalties.items():
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=Cs, cv=5, scoring="accuracy", max_iter=5000,
            random_state=RANDOM_STATE, **cfg
        ),
    )
    clf.fit(X, y)
    lr = clf.named_steps["logisticregressioncv"]
    cv_models[name] = clf
    print(name, "best C:", lr.C_[0], "nonzero:", np.sum(np.abs(lr.coef_[0]) > 1e-6))


Lasso sets coefficients exactly to zero and is most directly interpretable for variable selection. Ridge shrinks all coefficients but rarely selects variables. Elastic net is a compromise and tends to keep correlated variables together better than lasso. A cross-validated lambda optimizes prediction, not necessarily exact variable selection, so it can retain too many or too few genes.


## Task 2 - simulated lasso: PSR and FDR


In [ ]:
def simulate_logistic(n=300, p=20, link="logit", random_state=42):
    local_rng = np.random.default_rng(random_state)
    beta = np.r_[np.ones(10), np.zeros(p - 10)]
    X = local_rng.normal(size=(n, p))
    eta = X @ beta
    prob = expit(eta) if link == "logit" else norm.cdf(eta)
    y = local_rng.binomial(1, prob)
    return X, y, set(range(10))


def lasso_select(X, y):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            penalty="l1", solver="saga", Cs=np.logspace(-2, 1, 8),
            cv=3, max_iter=3000, random_state=RANDOM_STATE
        ),
    )
    clf.fit(X, y)
    coef = clf.named_steps["logisticregressioncv"].coef_[0]
    return set(np.flatnonzero(np.abs(coef) > 1e-6)), clf


def psr_fdr(selected, true_set):
    psr = len(selected & true_set) / len(true_set)
    fdr = len(selected - true_set) / max(len(selected), 1)
    return psr, fdr


X_sim, y_sim, true_set = simulate_logistic(n=300, p=20, random_state=1)
selected, clf = lasso_select(X_sim, y_sim)
print("selected variables:", sorted(i + 1 for i in selected))
print("PSR, FDR:", psr_fdr(selected, true_set))


In [ ]:
def simulation_grid(sample_sizes=(50, 100, 300, 500, 1000), irrelevant=(10, 50, 100), L=20, link="logit"):
    by_n = []
    for n in sample_sizes:
        vals = []
        for rep in range(L):
            X, y, true_set = simulate_logistic(n=n, p=20, link=link, random_state=1000 + rep + n)
            selected, _ = lasso_select(X, y)
            vals.append(psr_fdr(selected, true_set))
        by_n.append((n, *np.mean(vals, axis=0)))

    by_p = []
    for k_irrelevant in irrelevant:
        vals = []
        p = 10 + k_irrelevant
        for rep in range(L):
            X, y, true_set = simulate_logistic(n=300, p=p, link=link, random_state=2000 + rep + p)
            selected, _ = lasso_select(X, y)
            vals.append(psr_fdr(selected, true_set))
        by_p.append((k_irrelevant, *np.mean(vals, axis=0)))
    return np.array(by_n), np.array(by_p)


by_n, by_p = simulation_grid(L=10)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(by_n[:, 0], by_n[:, 1], marker="o", label="PSR")
axes[0].plot(by_n[:, 0], by_n[:, 2], marker="o", label="FDR")
axes[0].set_title("Selection quality vs sample size")
axes[1].plot(by_p[:, 0], by_p[:, 1], marker="o", label="PSR")
axes[1].plot(by_p[:, 0], by_p[:, 2], marker="o", label="FDR")
axes[1].set_title("Selection quality vs irrelevant variables")
for ax in axes:
    ax.grid(True)
    ax.legend()
plt.show()


In [ ]:
logit_n, _ = simulation_grid(sample_sizes=(100, 300, 500), irrelevant=(10,), L=10, link="logit")
probit_n, _ = simulation_grid(sample_sizes=(100, 300, 500), irrelevant=(10,), L=10, link="probit")
print("logit rows [n, PSR, FDR]:\n", logit_n)
print("probit rows [n, PSR, FDR]:\n", probit_n)


Prediction can be good even when variable selection is poor because correlated or proxy variables can predict well. PSR measures how many true signals are recovered; FDR measures how many selected variables are false discoveries. Lasso prediction is usually more robust to link misspecification than exact PSR/FDR.
